In [1]:
# Install specific versions
!pip install -q google-genai>=0.1.0 datasets>=2.14.0 pandas>=2.0.0 numpy<2.0.0 python-dotenv

# Check for dependency conflicts
!pip check

# System & GPU Diagnostics (Though API-bound, good to know system state)
!python --version
!nvidia-smi || echo "No GPU detected; running on CPU (Expected for API-bound tasks)."


/bin/bash: line 1: 2.0.0: No such file or directory
ipython 7.34.0 requires jedi, which is not installed.
Python 3.13.18
/bin/bash: line 1: nvidia-smi: command not found
No GPU detected; running on CPU (Expected for API-bound tasks).


In [2]:
import os
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from google import genai
from google.colab import userdata

# 1. Deterministic Control
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    print(f"[OK] Random seeds set to {seed}")

set_seeds(42)

# 2. Dataset Access Procedure
print("Testing Hugging Face Dataset access...")
try:
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    sample_q = ds[0]['question']
    print(f"[OK] Dataset loaded. Row 0 Question: '{sample_q[:50]}...'")
except Exception as e:
    print(f"[FAIL] Dataset load error: {e}")

# 3. API Diagnostic Procedure
print("Testing Google GenAI API Connection...")
try:
    api_key = userdata.get('secretName')
    client = genai.Client(api_key=api_key)

    # Minimal test call
    response = client.models.generate_content(
        model='gemini-3.8-flash',
        contents='Return the integer 1.'
    )

    print(f"[OK] API Connection successful. Response: {response.text.strip()}")
    if response.usage_metadata:
        print(f"[OK] Usage Metadata tracked: Input={response.usage_metadata.prompt_token_count}, Output={response.usage_metadata.candidates_token_count}")
    else:
        print("[WARN] Usage metadata missing from response.")
except Exception as e:
    print(f"[FAIL] API Error: {e}")

[OK] Random seeds set to 42
Testing Hugging Face Dataset access...


README.md:   0%|          | 0.00/5.19k [00:00<?, ?B/s]

pqa_labeled/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

pqa_labeled/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

[OK] Dataset loaded. Row 0 Question: 'Do mitochondria play a role in remodelling lace pl...'
Testing Google GenAI API Connection...
[OK] API Connection successful. Response: 1
[OK] Usage Metadata tracked: Input=7, Output=1


In [4]:
import pandas as pd
from datasets import load_dataset

# 1. Load the labeled partition
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
df = dataset.to_pandas()

# 2. Extract and format the necessary columns
# Join the contexts array into a single string to prevent RAG prompt bleed
df['joined_contexts'] = df['context'].apply(lambda x: "\n".join(x['context']) if isinstance(x, dict) and 'context' in x else "\n".join(x))

# 3. Create the 50-sample frozen split using the established random seed
# We use pandas sample for a reproducible random selection
frozen_eval_df = df[['pubid', 'question', 'joined_contexts', 'long_answer']].sample(n=50, random_state=42)

# 4. Save to CSV in the structured repository
import os
os.makedirs("results/data", exist_ok=True)
frozen_eval_df.to_csv("results/data/frozen_eval_split.csv", index=False)

print(f"[LOCKED] Saved {len(frozen_eval_df)} records to results/data/frozen_eval_split.csv")
print("First row question preview:", frozen_eval_df.iloc[0]['question'])

[LOCKED] Saved 50 records to results/data/frozen_eval_split.csv
First row question preview: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


In [1]:
!pip install datasets google-generativeai

In [3]:
import pandas as pd
import time
from google import genai
from google.colab import userdata

def generate_zeroshot_rationale(question: str, model_name: str = "gemini-3.8-flash") -> dict:
    """
    Generates a zero-shot scientific rationale using the new google.genai SDK.
    """
    # 1. Securely load the API key from Colab Secrets
    try:
        api_key = userdata.get('secretName')
    except userdata.SecretNotFoundError:
        return {
            "rationale": "ERROR: 'GEMINI_API_KEY' secret not found in Colab.",
            "input_tokens": 0,
            "output_tokens": 0,
            "status": "error"
        }

    # 2. Instantiate the new client
    client = genai.Client(api_key=api_key)

    system_instruction = (
        "You are an expert biomedical AI. Answer the following scientific question "
        "with a clear, highly plausible, and scientifically accurate rationale."
    )

    try:
        # Enforce rate-limit delay
        time.sleep(2)

        # 3. New SDK syntax for generation and configuration
        response = client.models.generate_content(
            model=model_name,
            contents=question,
            config=genai.types.GenerateContentConfig(
                system_instruction=system_instruction
            )
        )

        return {
            "rationale": response.text,
            "input_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "status": "success"
        }
    except Exception as e:
        return {
            "rationale": f"ERROR: {str(e)}",
            "input_tokens": 0,
            "output_tokens": 0,
            "status": "error"
        }

# ==========================================
# SANITY TEST
# ==========================================
print("Loading frozen dataset...")
df_frozen = pd.read_csv("results/data/frozen_eval_split.csv")
test_row = df_frozen.iloc[0]

print(f"Testing Zero-Shot Baseline on PubID: {test_row['pubid']}")
print(f"Question: {test_row['question']}\n")
print("Generating rationale (this may take a few seconds)...\n")

start_time = time.time()
result = generate_zeroshot_rationale(test_row['question'])
elapsed = time.time() - start_time

print(f"Status: {result['status']}")
print(f"Time Elapsed: {elapsed:.2f} seconds")
print(f"Input Tokens: {result['input_tokens']}")
print(f"Output Tokens: {result['output_tokens']}")
print("\nGenerated Rationale Preview:")
print("-" * 40)
print(result['rationale'][:500] + "..." if len(result['rationale']) > 500 else result['rationale'])
print("-" * 40)

Loading frozen dataset...
Testing Zero-Shot Baseline on PubID: 26348845
Question: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

Generating rationale (this may take a few seconds)...

Status: success
Time Elapsed: 211.22 seconds
Input Tokens: 42
Output Tokens: 529

Generated Rationale Preview:
----------------------------------------
**Yes, Pap smears containing glandular cell abnormalities can be detected by rapid prescreening (RPS), but they are detected with more variable and generally lower sensitivity compared to squamous cell abnormalities.**

### 1. Evidence and Feasibility of Detection
Studies evaluating rapid prescreening (typically a 30- to 90-second scan of the slide prior to routine evaluation) demonstrate that RPS can identify the majority of clinically significant cervical abnormalities, including:
* **Atypical...
----------------------------------------


In [8]:
def generate_rag_rationale(question: str, context: str, model_name: str = "gemini-3.6-flash") -> dict:
    """
    Generates a RAG-based scientific rationale using the provided context.
    """
    try:
        api_key = userdata.get('secretName2')
    except userdata.SecretNotFoundError:
        return {"rationale": "ERROR: API key not found.", "input_tokens": 0, "output_tokens": 0, "status": "error"}

    client = genai.Client(api_key=api_key)

    system_instruction = (
        "You are an expert biomedical AI. Answer the following scientific question "
        "using the provided context. Provide a clear, highly plausible, and "
        "scientifically accurate rationale based on the context."
    )

    # Strictly formatting the boundary between context and the question
    combined_prompt = f"Context:\n{context}\n\nQuestion:\n{question}"

    try:
        time.sleep(2) # Rate-limit delay

        response = client.models.generate_content(
            model=model_name,
            contents=combined_prompt,
            config=genai.types.GenerateContentConfig(
                system_instruction=system_instruction
            )
        )

        return {
            "rationale": response.text,
            "input_tokens": response.usage_metadata.prompt_token_count,
            "output_tokens": response.usage_metadata.candidates_token_count,
            "status": "success"
        }
    except Exception as e:
        return {
            "rationale": f"ERROR: {str(e)}",
            "input_tokens": 0,
            "output_tokens": 0,
            "status": "error"
        }

# ==========================================
# SANITY TEST
# ==========================================
print(f"Testing RAG Baseline on PubID: {test_row['pubid']}")
print(f"Question: {test_row['question']}")
print(f"Context Length: {len(test_row['joined_contexts'])} characters\n")
print("Generating RAG rationale...\n")

start_time = time.time()
rag_result = generate_rag_rationale(test_row['question'], test_row['joined_contexts'])
elapsed = time.time() - start_time

print(f"Status: {rag_result['status']}")
print(f"Time Elapsed: {elapsed:.2f} seconds")
print(f"Input Tokens: {rag_result['input_tokens']} (Expected to be higher than Zero-Shot)")
print(f"Output Tokens: {rag_result['output_tokens']}")
print("\nGenerated RAG Rationale Preview:")
print("-" * 40)
print(rag_result['rationale'][:500] + "..." if len(rag_result['rationale']) > 500 else rag_result['rationale'])
print("-" * 40)

Testing RAG Baseline on PubID: 26348845
Question: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?
Context Length: 66 characters

Generating RAG rationale...

Status: success
Time Elapsed: 12.56 seconds
Input Tokens: 77 (Expected to be higher than Zero-Shot)
Output Tokens: 361

Generated RAG Rationale Preview:
----------------------------------------
Based on clinical cytology literature and evidence regarding cervical cancer screening quality assurance methods, **yes, Pap smears with glandular cell abnormalities can be detected by rapid prescreening (RPS).**

### Scientific Rationale:

1. **Cellular and Architectural Features:** 
   Glandular cell abnormalities—such as atypical glandular cells (AGC), adenocarcinoma in situ (AIS), and invasive adenocarcinoma—typically form distinct, three-dimensional, hyperchromatic crowded groups (HCGs). Fe...
----------------------------------------


In [9]:
import pandas as pd
import time
# Using tqdm for a progress bar so you can see the loop advancing
from tqdm.auto import tqdm

# Ensure the results directory exists
import os
os.makedirs("results/data", exist_ok=True)

# Load the frozen dataset if not already in memory
df_frozen = pd.read_csv("results/data/frozen_eval_split.csv")

batch_results = []
print("Starting batch processing for 50 samples. This will take a few minutes...")

for index, row in tqdm(df_frozen.iterrows(), total=len(df_frozen)):
    question = row['question']
    context = row['joined_contexts']
    pubid = row['pubid']
    reference = row['long_answer']

    # 1. Run Zero-Shot Baseline
    zs_result = generate_zeroshot_rationale(question, model_name="gemini-3.6-flash")

    # 2. Run RAG Baseline
    rag_result = generate_rag_rationale(question, context, model_name="gemini-3.6-flash")

    # 3. Store the data
    batch_results.append({
        "pubid": pubid,
        "question": question,
        "reference_answer": reference,
        "zs_rationale": zs_result['rationale'],
        "zs_input_tokens": zs_result['input_tokens'],
        "zs_output_tokens": zs_result['output_tokens'],
        "zs_status": zs_result['status'],
        "rag_rationale": rag_result['rationale'],
        "rag_input_tokens": rag_result['input_tokens'],
        "rag_output_tokens": rag_result['output_tokens'],
        "rag_status": rag_result['status']
    })

# Convert to DataFrame and save
results_df = pd.DataFrame(batch_results)
results_df.to_csv("results/data/baseline_results.csv", index=False)

print(f"\n[SUCCESS] Saved {len(results_df)} processed rows to results/data/baseline_results.csv")

Starting batch processing for 50 samples. This will take a few minutes...


  0%|          | 0/50 [00:00<?, ?it/s]


[SUCCESS] Saved 50 processed rows to results/data/baseline_results.csv


In [15]:
import pandas as pd
import time
import random
import re
from google import genai
from google.colab import userdata
from tqdm.auto import tqdm

def evaluate_rationale_with_retry(question: str, reference: str, rationale: str, model_name: str = "gemini-3.5-flash-lite", max_retries: int = 6) -> int:
    """Scores a rationale from 1-10, with exponential backoff for API capacity errors."""
    try:
        api_key = userdata.get('secretName3')
        client = genai.Client(api_key=api_key)
    except Exception:
        print("API Key error.")
        return 0

    system_instruction = (
        "You are an impartial expert biomedical judge. Evaluate the generated scientific rationale "
        "against the provided reference answer. Score the plausibility and scientific accuracy of the "
        "rationale on a scale of 1 to 10. Output ONLY a single integer from 1 to 10. Do not include any other text."
    )
    prompt = f"Question: {question}\nReference Answer: {reference}\nGenerated Rationale: {rationale}"

    for attempt in range(max_retries):
        try:
            # Base delay to respect standard rate limits
            time.sleep(2)

            response = client.models.generate_content(
                model=model_name,
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.0
                )
            )

            match = re.search(r'\b(10|[1-9])\b', response.text)
            return int(match.group(1)) if match else 0

        except Exception as e:
            error_str = str(e).upper()
            # Catch server capacity (503) and rate limit (429) errors
            if "503" in error_str or "UNAVAILABLE" in error_str or "429" in error_str or "QUOTA" in error_str:
                # Exponential backoff: 4s, 8s, 16s, 32s... plus a tiny bit of random jitter
                wait_time = (2 ** (attempt + 1)) * 2 + random.uniform(0, 1)
                print(f"\n[Server Busy] 503/429 Error on attempt {attempt + 1}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)
            else:
                print(f"\n[Fatal Error] Unresolvable error: {e}")
                return 0

    print("\n[Failed] Max retries reached for this rationale. Assigning score of 0.")
    return 0

# 1. Load the generated baseline results
df_results = pd.read_csv("results/data/baseline_results.csv")

zs_scores = []
rag_scores = []

print(f"Scoring baselines with LLM-as-a-Judge (gemini-3.5-flash-lite) with exponential backoff...")
for index, row in tqdm(df_results.iterrows(), total=len(df_results)):

    # Score Zero-Shot
    zs_score = evaluate_rationale_with_retry(row['question'], row['reference_answer'], str(row['zs_rationale']))
    zs_scores.append(zs_score)

    # Score RAG
    rag_score = evaluate_rationale_with_retry(row['question'], row['reference_answer'], str(row['rag_rationale']))
    rag_scores.append(rag_score)

df_results['zs_plausibility'] = zs_scores
df_results['rag_plausibility'] = rag_scores

# Save the fully scored baselines
df_results.to_csv("results/data/scored_baseline_results.csv", index=False)

# 2. Calculate and Display the Final Baseline Metrics
metrics = {
    "Architecture": ["Zero-Shot Single LLM", "Static Context RAG"],
    "Mean Plausibility (1-10)": [df_results['zs_plausibility'].mean(), df_results['rag_plausibility'].mean()],
    "Total Tokens Consumed": [
        df_results['zs_input_tokens'].sum() + df_results['zs_output_tokens'].sum(),
        df_results['rag_input_tokens'].sum() + df_results['rag_output_tokens'].sum()
    ]
}

metrics_df = pd.DataFrame(metrics)
metrics_df['Plausibility ROI (Score per 1k Tokens)'] = metrics_df['Mean Plausibility (1-10)'] / (metrics_df['Total Tokens Consumed'] / 1000)

print("\n[LOCKED] Baseline Evaluation Complete. Final Metrics:")
print("-" * 60)
print(metrics_df.to_string(index=False))
print("-" * 60)

Scoring baselines with LLM-as-a-Judge (gemini-3.5-flash-lite) with exponential backoff...


  0%|          | 0/50 [00:00<?, ?it/s]


[Server Busy] 503/429 Error on attempt 1. Retrying in 4.2 seconds...

[LOCKED] Baseline Evaluation Complete. Final Metrics:
------------------------------------------------------------
        Architecture  Mean Plausibility (1-10)  Total Tokens Consumed  Plausibility ROI (Score per 1k Tokens)
Zero-Shot Single LLM                      1.54                   2502                                0.615508
  Static Context RAG                      2.14                   3320                                0.644578
------------------------------------------------------------


In [16]:
import pandas as pd
import time
from google import genai
from google.colab import userdata

def generate_debate_rationale(question: str, model_name: str = "gemini-3.5-flash-lite", rounds: int = 1) -> dict:
    """
    Executes a heterogeneous 2-agent debate to generate a scientific rationale.
    Tracks cumulative token costs across all API calls.
    """
    try:
        api_key = userdata.get('secretName3')
        client = genai.Client(api_key=api_key)
    except Exception:
        return {"rationale": "ERROR: API key not found.", "input_tokens": 0, "output_tokens": 0, "status": "error"}

    # Persona 1: The Proposer (Focuses on generating the initial hypothesis)
    proposer_instruction = (
        "You are an expert biomedical researcher. Formulate a highly plausible, "
        "scientifically accurate rationale answering the user's question. "
        "Defend your position with logical scientific principles."
    )

    # Persona 2: The Challenger (Focuses on finding flaws and sycophancy)
    challenger_instruction = (
        "You are a rigorous scientific peer reviewer. Critique the proposed biomedical rationale. "
        "Identify logical fallacies, lack of evidence, or sycophantic assumptions. "
        "Provide a counter-argument or suggest necessary refinements."
    )

    total_input_tokens = 0
    total_output_tokens = 0

    try:
        # STEP 1: Proposer generates initial rationale
        time.sleep(2)
        proposer_response = client.models.generate_content(
            model=model_name,
            contents=question,
            config=genai.types.GenerateContentConfig(system_instruction=proposer_instruction)
        )
        current_rationale = proposer_response.text
        total_input_tokens += proposer_response.usage_metadata.prompt_token_count
        total_output_tokens += proposer_response.usage_metadata.candidates_token_count

        # STEP 2: Debate Loop
        for r in range(rounds):
            # Challenger critiques the Proposer's rationale
            time.sleep(2)
            critique_prompt = f"Original Question: {question}\n\nProposed Rationale: {current_rationale}\n\nProvide your critique:"
            challenger_response = client.models.generate_content(
                model=model_name,
                contents=critique_prompt,
                config=genai.types.GenerateContentConfig(system_instruction=challenger_instruction)
            )
            critique = challenger_response.text
            total_input_tokens += challenger_response.usage_metadata.prompt_token_count
            total_output_tokens += challenger_response.usage_metadata.candidates_token_count

            # Proposer revises based on critique
            time.sleep(2)
            revision_prompt = f"Original Question: {question}\n\nYour Previous Rationale: {current_rationale}\n\nPeer Review Critique: {critique}\n\nRevise your rationale to address the critique and provide the final consensus answer."
            revision_response = client.models.generate_content(
                model=model_name,
                contents=revision_prompt,
                config=genai.types.GenerateContentConfig(system_instruction=proposer_instruction)
            )
            current_rationale = revision_response.text
            total_input_tokens += revision_response.usage_metadata.prompt_token_count
            total_output_tokens += revision_response.usage_metadata.candidates_token_count

        return {
            "rationale": current_rationale,
            "input_tokens": total_input_tokens,
            "output_tokens": total_output_tokens,
            "status": "success"
        }

    except Exception as e:
        return {
            "rationale": f"ERROR: {str(e)}",
            "input_tokens": total_input_tokens,
            "output_tokens": total_output_tokens,
            "status": "error"
        }

# ==========================================
# SANITY TEST
# ==========================================
print("Loading frozen dataset...")
df_frozen = pd.read_csv("results/data/frozen_eval_split.csv")
test_row = df_frozen.iloc[0]

print(f"Testing 2-Agent Debate on PubID: {test_row['pubid']}")
print(f"Question: {test_row['question']}\n")
print("Executing debate (this involves multiple API calls and will take 10+ seconds)...\n")

start_time = time.time()
debate_result = generate_debate_rationale(test_row['question'], model_name="gemini-3.5-flash-lite", rounds=1)
elapsed = time.time() - start_time

print(f"Status: {debate_result['status']}")
print(f"Time Elapsed: {elapsed:.2f} seconds")
print(f"Total Input Tokens: {debate_result['input_tokens']} (Expected to be high due to loop)")
print(f"Total Output Tokens: {debate_result['output_tokens']}")
print("\nFinal Consensus Rationale Preview:")
print("-" * 40)
print(debate_result['rationale'][:500] + "..." if len(debate_result['rationale']) > 500 else debate_result['rationale'])
print("-" * 40)

Loading frozen dataset...
Testing 2-Agent Debate on PubID: 26348845
Question: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

Executing debate (this involves multiple API calls and will take 10+ seconds)...

Status: success
Time Elapsed: 29.03 seconds
Total Input Tokens: 2799 (Expected to be high due to loop)
Total Output Tokens: 2680

Final Consensus Rationale Preview:
----------------------------------------
To answer whether Pap smears with glandular cell abnormalities are detected by rapid prescreening, we must examine the biological nature of glandular lesions, the operational mechanics of rapid prescreening (RPS), and the distinct limitations of human visual search patterns under time constraints. 

**The short answer is no; human rapid prescreening is largely ineffective and unsafe for the reliable detection of glandular cell abnormalities.** 

Here is the scientifically refined and peer-review...
----------------------------------------


In [17]:
import pandas as pd
import time
import random
from tqdm.auto import tqdm

# 1. Load frozen split and previously scored baselines
df_frozen = pd.read_csv("results/data/frozen_eval_split.csv")
df_baselines = pd.read_csv("results/data/scored_baseline_results.csv")

debate_results = []

def robust_debate_generation(question, max_retries=5):
    """Wraps the debate generation with exponential backoff for 503/429 errors."""
    for attempt in range(max_retries):
        try:
            res = generate_debate_rationale(question, model_name="gemini-3.5-flash-lite", rounds=1)
            if "ERROR" in res['rationale']:
                raise Exception(res['rationale'])
            return res
        except Exception as e:
            wait_time = (2 ** (attempt + 1)) + random.uniform(0, 1)
            print(f"API busy, retrying debate in {wait_time:.1f}s...")
            time.sleep(wait_time)
    return {"rationale": "FAILED", "input_tokens": 0, "output_tokens": 0, "status": "error"}

print("Executing Heterogeneous 2-Agent Debate Pipeline...")
for index, row in tqdm(df_frozen.iterrows(), total=len(df_frozen)):
    # Run the debate loop
    debate_res = robust_debate_generation(row['question'])

    # Score the consensus using our previously defined judge with backoff
    score = evaluate_rationale_with_retry(
        row['question'],
        row['long_answer'],
        debate_res['rationale'],
        model_name="gemini-3.5-flash-lite"
    )

    debate_results.append({
        "debate_rationale": debate_res['rationale'],
        "debate_input_tokens": debate_res['input_tokens'],
        "debate_output_tokens": debate_res['output_tokens'],
        "debate_plausibility": score
    })

# Merge and save final dataset
df_debate = pd.DataFrame(debate_results)
df_final = pd.concat([df_baselines, df_debate], axis=1)
df_final.to_csv("results/data/final_experiment_results.csv", index=False)

# 2. Calculate Final Comparative Metrics
metrics = {
    "Architecture": ["Zero-Shot Single LLM", "Static Context RAG", "2-Agent Debate (Ungrounded)"],
    "Mean Plausibility (1-10)": [
        df_final['zs_plausibility'].mean(),
        df_final['rag_plausibility'].mean(),
        df_final['debate_plausibility'].mean()
    ],
    "Total Tokens Consumed": [
        df_final['zs_input_tokens'].sum() + df_final['zs_output_tokens'].sum(),
        df_final['rag_input_tokens'].sum() + df_final['rag_output_tokens'].sum(),
        df_final['debate_input_tokens'].sum() + df_final['debate_output_tokens'].sum()
    ]
}

metrics_df = pd.DataFrame(metrics)
metrics_df['Plausibility ROI (Score per 1k Tokens)'] = metrics_df['Mean Plausibility (1-10)'] / (metrics_df['Total Tokens Consumed'] / 1000)

print("\n[LOCKED] Full Experiment Complete. Final Metrics:")
print("-" * 80)
print(metrics_df.to_string(index=False))
print("-" * 80)

Executing Heterogeneous 2-Agent Debate Pipeline...


  0%|          | 0/50 [00:00<?, ?it/s]


[Server Busy] 503/429 Error on attempt 1. Retrying in 4.9 seconds...

[Server Busy] 503/429 Error on attempt 1. Retrying in 4.7 seconds...
API busy, retrying debate in 2.4s...

[LOCKED] Full Experiment Complete. Final Metrics:
--------------------------------------------------------------------------------
               Architecture  Mean Plausibility (1-10)  Total Tokens Consumed  Plausibility ROI (Score per 1k Tokens)
       Zero-Shot Single LLM                      1.54                   2502                                0.615508
         Static Context RAG                      2.14                   3320                                0.644578
2-Agent Debate (Ungrounded)                      8.58                 294079                                0.029176
--------------------------------------------------------------------------------
